# Integrated Gradients vs Gradient × Input for scoring formulae

Instance rules pick their candidate formulae by an attribution score $s_j$ (`ir.candidate_scores`). With logits $z = Wx$ (no intercept) and predicted class $k$, the score explains the log-odds of $k$

$$g(x) = z_k - \log \sum_{l \ne k} e^{z_l}, \qquad \frac{\partial g}{\partial x_j} = \sum_{l \ne k} \pi_l(x)\,(w_{kj} - w_{lj}), \qquad \pi(x) = \operatorname{softmax}_{l \ne k}(z_l) = \frac{P(l \mid x)}{1 - P(k \mid x)}.$$

Both methods write $s_j$ as a convex combination of the margin attributions $A_{lj} = (w_{kj} - w_{lj})\,x_j$ (eq. 17); only the weights differ:

| method | $s_j$ | weights |
|---|---|---|
| Gradient × Input (current) | $\sum_l \pi_l(x)\,A_{lj}$ | $\pi(x)$: softmax of the other classes' logits at $x$ |
| Integrated Gradients, baseline $x' = 0$ | $\sum_l \bar\pi_l\,A_{lj}$ | $\bar\pi = \int_0^1 \pi(\alpha x)\,d\alpha = \int_0^1 \operatorname{softmax}_{l \ne k}(\alpha z_l)\,d\alpha$ |

The baseline $x' = 0$ is the train mean of every formula (features are standardized), the same reference the literals are built around. Along the path the weights go from uniform ($\alpha = 0$) to $\pi(x)$ ($\alpha = 1$), so IG spreads the score over all competing classes, while G×I concentrates it on the closest competitors.

- **Completeness**: $\sum_j \mathrm{IG}_j = g(x) - g(0) = g(x) + \log(K - 1)$ (checked below, it measures the quadrature error).
- **Binary problems** ($K = 2$): $\pi = \bar\pi = 1$, the two methods coincide.

The rule search uses the scores in two places: the candidate set $J = \{j : s_j > 0\}$, and the tie-break between extensions with the same bound and TP. Everything else is unchanged.

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, for the local stlrocket package

from contextlib import contextmanager
from functools import reduce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from torcheck.stl import Or

from stlrocket.config import ExperimentConfig
from stlrocket.data import load_dataset
from stlrocket.features import build_formula_bank, eval_robustness
from stlrocket.classifier import train_classifier
from stlrocket.formula_sampler import F0
from stlrocket import instance_rules as ir

# Data and model (multi-class: with two classes the methods coincide)
DATASET = "Libras"
N_FORMULAS = 1000
DEPTH = 2
SEED = 0

# Instance rules and global explanations
RULES = ir.RuleConfig(n_max=3, beam_width=5, confidence=0.9, min_new_share=0.05)

# Integrated Gradients: Gauss-Legendre nodes on [0, 1] for the path integral of the weights
N_NODES = 128

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## 1. Train the model
As in `explanations.ipynb`: formula bank + lasso logistic regression without intercept.

In [ ]:
X_tr, y_tr_raw, X_te, y_te_raw = load_dataset(DATASET)
cfg = ExperimentConfig(
    dataset=DATASET, n_formulas=N_FORMULAS, depth_max=DEPTH, until_weight=0.0, cv=3, cut_point=1.0,
    fit_intercept=False, n_run=1, base_seed=SEED, output_dir=".", device="cpu",
    # fields of the old explanation pipeline, unused here
    explain=False, pool_size=0, precision_threshold=0.0, simplify_agreement=0.0, simplify_min_gain=0.0,
    simplify_decimals=0,
)
formulas, F_tr, F_te, mu, sigma = build_formula_bank(X_tr, X_te, cfg, SEED)
model = train_classifier(F_tr, y_tr_raw, cfg, seed=SEED)
W = ir.model_logits_matrix(model)
classes = model.classes_
index = {c: i for i, c in enumerate(classes)}

SPLITS = ("train", "test")
F = {"train": F_tr, "test": F_te}
X = {"train": X_tr, "test": X_te}
pred = {s: ir.predict_indices(F[s], W) for s in SPLITS}
true = {"train": np.array([index[c] for c in y_tr_raw]), "test": np.array([index[c] for c in y_te_raw])}

print(f"{DATASET}: {len(X_tr)} train / {len(X_te)} test series, {len(classes)} classes")
print(f"formulae used by the model: {int(np.any(W != 0, axis=0).sum())} of {N_FORMULAS}")
for s in SPLITS:
    print(f"{s} accuracy: {(pred[s] == true[s]).mean():.3f}")

## 2. The two scores
`scores_from(weights)` builds a drop-in replacement of `ir.candidate_scores` from a function giving the weights over the other classes; `gxi_weights` reproduces the current scores (checked against `ir.candidate_scores`), `ig_weights` integrates them along the path $\alpha x$, $\alpha \in [0, 1]$.

In [ ]:
_nodes, _w = np.polynomial.legendre.leggauss(N_NODES)
ALPHA, ALPHA_W = (_nodes + 1) / 2, _w / 2          # Gauss-Legendre on [0, 1]


def softmax_rows(Z):
    E = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)


def gxi_weights(z_others):
    # pi(x) = softmax of the other classes' logits = P(l|x) / (1 - P(k|x))
    return softmax_rows(z_others)


def ig_weights(z_others):
    # pi_bar = int_0^1 softmax(alpha * z_others) d alpha (path from the baseline x' = 0 to x)
    return ALPHA_W @ softmax_rows(ALPHA[:, None] * z_others[None, :])


def scores_from(weights):
    # Same contract as ir.candidate_scores: predicted class k, scores s, candidates J ranked by decreasing s
    def candidate_scores(x, W_full):
        z = W_full @ x
        k = int(np.argmax(z))
        others = np.arange(len(z)) != k
        A = (W_full[k] - W_full[others]) * x        # (K-1, M) margin attributions, eq. 17
        s = weights(z[others]) @ A
        J = np.flatnonzero(s > 0)
        return k, s, J[np.argsort(-s[J], kind="stable")]
    return candidate_scores


METHODS = {"Gradient x Input": scores_from(gxi_weights), "Integrated Gradients": scores_from(ig_weights)}

# Sanity check: the G x I replacement gives the scores of the package
err = max(np.abs(ir.candidate_scores(x, W)[1] - METHODS["Gradient x Input"](x, W)[1]).max() for x in F_te)
print(f"G x I vs ir.candidate_scores, max abs difference of the scores: {err:.2e}")

# Completeness of IG: sum_j IG_j = g(x) - g(0) = g(x) + log(K - 1)
def log_odds(z, k):
    zo = np.delete(z, k)
    return z[k] - (zo.max() + np.log(np.exp(zo - zo.max()).sum()))

gap = []
for x in F_te:
    k, s, _ = METHODS["Integrated Gradients"](x, W)
    z = W @ x
    gap.append(abs(s.sum() - (log_odds(z, k) + np.log(len(z) - 1))) / max(abs(log_odds(z, k)), 1e-12))
print(f"IG completeness, relative error over the test series: max {max(gap):.2e}, median {np.median(gap):.2e}")

## 3. How different are the scores?
Per test series: total variation distance between the weights $\pi(x)$ and $\bar\pi$, overlap of the candidate sets $J$, overlap of the top-3 candidates (`n_max` = 3), and Spearman correlation of the scores of the formulae used by the model.

In [ ]:
used = np.flatnonzero(np.any(W != 0, axis=0))
rows = []
for i, x in enumerate(F_te):
    z = W @ x
    k = int(np.argmax(z))
    zo = np.delete(z, k)
    (_, s_g, J_g), (_, s_i, J_i) = (m(x, W) for m in METHODS.values())
    rows.append({
        "TV(pi, pi_bar)": 0.5 * np.abs(gxi_weights(zo) - ig_weights(zo)).sum(),
        "max pi": gxi_weights(zo).max(),
        "Jaccard J": len(set(J_g) & set(J_i)) / max(len(set(J_g) | set(J_i)), 1),
        "|J| G x I": len(J_g), "|J| IG": len(J_i),
        "top-3 overlap": len(set(J_g[:3]) & set(J_i[:3])) / 3,
        "Spearman": spearmanr(s_g[used], s_i[used]).statistic,
    })
cmp = pd.DataFrame(rows)
display(cmp.describe().loc[["mean", "min", "50%", "max"]].round(3))

fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
axs[0].scatter(cmp["max pi"], cmp["TV(pi, pi_bar)"], s=12, color="#2a78d6")
axs[0].set(xlabel="largest G x I weight max pi(x)", ylabel="TV(pi, pi_bar)", title="weights")
axs[1].hist(cmp["Jaccard J"], bins=20, color="#2a78d6")
axs[1].set(xlabel="Jaccard of the candidate sets", title="candidates")
axs[2].hist(cmp["Spearman"], bins=20, color="#2a78d6")
axs[2].set(xlabel="Spearman of the scores", title="ranking")
for ax in axs:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## 4. Instance rules and global explanations with each score
The rule search calls `ir.candidate_scores`; `scoring(...)` swaps it for the duration of a block. Rules are evaluated as their simplified STL formula, metrics w.r.t. the model's predictions (the explained series left out on its own split), as in `explanations.ipynb`.

In [ ]:
@contextmanager
def scoring(candidate_scores):
    old = ir.candidate_scores
    ir.candidate_scores = candidate_scores
    try:
        yield
    finally:
        ir.candidate_scores = old


def or_mask(phis, Xs):
    return reduce(np.logical_or, [eval_robustness(p, Xs) > 0 for p in phis])


results, all_rules = [], {}
for name, fn in METHODS.items():
    with scoring(fn):
        R = {"train": ir.explain_instances(F_tr, W, F_tr, pred["train"], RULES, train=True),
             "test": ir.explain_instances(F_te, W, F_tr, pred["train"], RULES)}
        G = ir.global_explanations(R["train"], F_tr, X_tr, pred["train"], formulas, mu, sigma, RULES)
    all_rules[name] = R
    row = {"method": name}
    for own in SPLITS:   # rules of the series of `own`, evaluated on `own`
        found = [(i, r) for i, r in enumerate(R[own]) if r["literals"]]
        m = []
        for i, r in found:
            phi = ir.rule_formula(r["literals"], formulas, mu, sigma, X_tr, RULES)
            m.append({**ir.rule_metrics(eval_robustness(phi, X[own]) > 0, r["k"], pred[own], exclude=i),
                      "length": len(r["literals"]), "size": F0.formula_size(phi)})
        m = pd.DataFrame(m)
        row.update({f"local prec {own}": m.precision.mean(), f"local recall {own}": m.coverage.mean()})
        if own == "test":
            row.update({"local length": m.length.mean(), "local size": m["size"].mean()})
    g = pd.DataFrame([ir.rule_metrics(or_mask(phis, X_te), k, pred["test"]) for k, phis in G.items()])
    P, Rc = g.precision.fillna(0), g.coverage
    row.update({"global prec test": P.mean(), "global recall test": Rc.mean(),
                "global F1 test": (2 * P * Rc / (P + Rc)).fillna(0).mean(),
                "global rules/class": np.mean([len(p) for p in G.values()]),
                "global size/class": np.mean([sum(F0.formula_size(p) for p in phis) for phis in G.values()])})
    results.append(row)

display(pd.DataFrame(results).set_index("method").T.round(3))

same = np.mean([a["literals"] == b["literals"] for a, b in zip(*(all_rules[n]["test"] for n in METHODS))])
print(f"test series whose instance rule is identical under the two scores: {same:.1%}")